# Upload Kaggle Dataset Files to Google Cloud Storage

This notebook uploads Kaggle-mounted dataset files to Google Cloud Storage using the same pipeline as `scripts/upload_kaggle_to_gcs.py`.

The run flow is:

1. Select a configured Kaggle source.
2. Scan files from the Kaggle input mount.
3. Detect each file batch from its relative path.
4. Normalize paths before building GCS keys.
5. Write manifest, metrics, errors, and summary artifacts.
6. Run a dry run first, then switch to a real upload when the planned keys look correct.

Start with `DRY_RUN = True` and a small `MAX_FILES` value. Inspect the generated manifest before setting `DRY_RUN = False`.

## 1. Install Dependencies

Install the GCS client and progress-bar package used by the upload pipeline. Kaggle often already has `tqdm`, but installing it here keeps the notebook self-contained.

In [ ]:
# Kaggle usually has tqdm. Install GCS client if it is missing.
%pip install -q google-cloud-storage tqdm

## 2. Import Libraries

Load the Python modules used for path handling, file scanning, JSON/CSV artifacts, logging, MIME type detection, parallel uploads, timestamps, and lightweight runtime objects.

In [ ]:
import copy
import csv
import fnmatch
import json
import logging
import mimetypes
import os
import re
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from types import SimpleNamespace
from typing import Any


## 3. Configure Sources and Runtime Settings

Define the available Kaggle sources, valid batches, batch detection patterns, and GCS artifact prefixes.

For `l21_l30_ai_challenge_2025`, `relative_path_prefixes_to_strip` removes extra wrapper folders such as `ai-challenge-2025/Videos/Videos` or `Videos/Videos`. This keeps uploaded object keys from containing duplicate `Videos` folders.

The main values to edit for a run are `SOURCE_ID`, `BATCHES`, `GCS_BUCKET`, `MAX_FILES`, `DRY_RUN`, and `WORKERS`.

In [ ]:
# Edit this cell for normal runs. The dataset config mirrors configs/data_ingestion_sources.yaml.
CONFIG = {
    'schema_version': 1,
    'defaults': {
        'source_type': 'kaggle',
        'source_version': 'kaggle_current',
        'include_patterns': ['**/*.mp4', '**/*.avi', '**/*.mov', '**/*.mkv', '**/*.zip'],
        'exclude_patterns': ['**/.DS_Store', '**/__MACOSX/**'],
        'gcs': {
            'raw_prefix': 'raw/source=kaggle',
            'control_prefix': 'manifests/pipeline=kaggle_ingest',
            'logs_prefix': 'logs/pipeline=kaggle_ingest',
            'quarantine_prefix': 'quarantine',
        },
    },
    'datasets': [
        {
            'source_id': 'l21_l30_ai_challenge_2025',
            'display_name': 'AI Challenge 2025 - L21 to L30',
            'source_type': 'kaggle',
            'enabled': True,
            'dataset_ref': 'aresusayhi/ai-challenge-2025',
            'dataset_url': 'https://www.kaggle.com/datasets/aresusayhi/ai-challenge-2025',
            'kaggle_mount_path': '/kaggle/input/ai-challenge-2025',
            'dataset_id': 'ai_challenge_2025',
            'relative_path_prefixes_to_strip': ['ai-challenge-2025/Videos/Videos', 'Videos/Videos'],
            'expected_batches': ['L21', 'L22', 'L23', 'L24', 'L25', 'L26', 'L27', 'L28', 'L29', 'L30'],
            'batch_detection': {
                'strategy': 'regex',
                'pattern': r'(?i)(?:^|[/_.-])(L2[1-9]|L30)(?:[/_.-]|$)',
            },
        },
        {
            'source_id': 'k01_k10_data_video_batch_2_1',
            'display_name': 'Data Video Batch 2.1 - K01 to K10',
            'source_type': 'kaggle',
            'enabled': True,
            'dataset_ref': 'tuktuai/data-video-batch-2-1',
            'dataset_url': 'https://www.kaggle.com/datasets/tuktuai/data-video-batch-2-1',
            'kaggle_mount_path': '/kaggle/input/data-video-batch-2-1',
            'dataset_id': 'data_video_batch_2_1',
            'expected_batches': ['K01', 'K02', 'K03', 'K04', 'K05', 'K06', 'K07', 'K08', 'K09', 'K10'],
            'batch_detection': {
                'strategy': 'regex',
                'pattern': r'(?i)(?:^|[/_.-])(K0[1-9]|K10)(?:[/_.-]|$)',
            },
        },
        {
            'source_id': 'k11_k20_data_video_batch_2_2',
            'display_name': 'Data Video Batch 2.2 - K11 to K20',
            'source_type': 'kaggle',
            'enabled': True,
            'dataset_ref': 'tuktuai/data-video-batch2-2',
            'dataset_url': 'https://www.kaggle.com/datasets/tuktuai/data-video-batch2-2',
            'kaggle_mount_path': '/kaggle/input/data-video-batch2-2',
            'dataset_id': 'data_video_batch2_2',
            'expected_batches': ['K11', 'K12', 'K13', 'K14', 'K15', 'K16', 'K17', 'K18', 'K19', 'K20'],
            'batch_detection': {
                'strategy': 'regex',
                'pattern': r'(?i)(?:^|[/_.-])(K1[1-9]|K20)(?:[/_.-]|$)',
            },
        },
    ],
}

# Runtime settings. For real uploads, set GCS_BUCKET here or as a Kaggle Secret.
SOURCE_ID = 'l21_l30_ai_challenge_2025'
BATCHES = 'all'  # Examples: 'all', 'L21,L22', ['K01', 'K02']
INPUT_ROOT = None  # None uses the selected source's kaggle_mount_path.
GCS_BUCKET = ''  # Bucket name only, or gs://bucket/raw-prefix. Fallback: Kaggle Secret/env GCS_BUCKET.
GCS_PREFIX = ''  # Raw prefix inside the bucket. Empty uses prefix from GCS_BUCKET URI or CONFIG default.
GCS_CREDENTIALS_FILE = ''  # Optional path to service-account JSON.
SOURCE_VERSION = ''  # Empty uses CONFIG default source_version.
WORKERS = 4
RUN_ID = ''
RUN_DIR = Path('ingestion_runs')
MAX_FILES = None  # Set to 1 for a smoke test.
DRY_RUN = True
SKIP_EXISTING = True
OVERWRITE = False
UPLOAD_RUN_ARTIFACTS = True
FAIL_ON_UNMAPPED = False
NO_PROGRESS = False
VERBOSE = False


## 4. Define Config, Batch, and Path Helpers

Prepare helper functions that merge default settings into each source, select the requested source, parse batch filters, match include/exclude patterns, detect batch IDs with regex, and build GCS object keys.

The key path rule lives in `normalize_relative_path()`: the original Kaggle path is retained in `source_relative_path` for debugging, while the stripped path is used as `relative_path` for the GCS destination.

In [ ]:
@dataclass(frozen=True)
class RunPaths:
    run_dir: Path
    manifest: Path
    summary: Path
    errors: Path
    metrics: Path
    log: Path


@dataclass(frozen=True)
class UploadResult:
    status: str
    bytes_uploaded: int
    duration_ms: int
    error: str = ''
    generation: str = ''


def utc_now_iso() -> str:
    return datetime.now(UTC).isoformat(timespec='seconds').replace('+00:00', 'Z')


def new_run_id() -> str:
    stamp = datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
    return f'{stamp}_{uuid.uuid4().hex[:8]}'


def normalize_prefix(prefix: str) -> str:
    return prefix.strip().strip('/')


def deep_merge(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    result = copy.deepcopy(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result


def merged_sources(config: dict[str, Any]) -> list[dict[str, Any]]:
    defaults = config.get('defaults') or {}
    sources = []
    for source in config.get('datasets') or []:
        merged = deep_merge(defaults, source)
        merged['gcs'] = deep_merge(defaults.get('gcs', {}), source.get('gcs', {}))
        sources.append(merged)
    return sources


def list_sources(config: dict[str, Any] = CONFIG) -> None:
    for source in merged_sources(config):
        enabled = 'enabled' if source.get('enabled', True) else 'disabled'
        batches = ','.join(source.get('expected_batches') or [])
        mount = source.get('kaggle_mount_path', '')
        print(f"{source.get('source_id')} [{enabled}] batches={batches} mount={mount}")


def select_source(config: dict[str, Any], source_id: str) -> dict[str, Any]:
    matches = [src for src in merged_sources(config) if src.get('source_id') == source_id]
    if not matches:
        known = ', '.join(src.get('source_id', '<missing>') for src in merged_sources(config))
        raise ValueError(f"Unknown source_id '{source_id}'. Known sources: {known}")
    source = matches[0]
    if not source.get('enabled', True):
        raise ValueError(f'Source is disabled: {source_id}')
    return source


def parse_batches(raw: str | list[str] | tuple[str, ...] | set[str], expected_batches: list[str]) -> set[str]:
    expected = {batch.upper() for batch in expected_batches}
    if isinstance(raw, (list, tuple, set)):
        selected = {str(part).strip().upper() for part in raw if str(part).strip()}
    elif str(raw).strip().lower() in {'', 'all', '*'}:
        return expected
    else:
        selected = {part.strip().upper() for part in str(raw).split(',') if part.strip()}
    unknown = selected - expected
    if unknown:
        raise ValueError(f"Unknown batch(es): {', '.join(sorted(unknown))}. Expected: {', '.join(sorted(expected))}")
    return selected


def matches_any(rel_path: str, patterns: list[str]) -> bool:
    rel_lower = rel_path.lower()
    return any(fnmatch.fnmatchcase(rel_lower, pattern.lower()) for pattern in patterns)


def detect_batch(rel_path: str, source: dict[str, Any]) -> str | None:
    detection = source.get('batch_detection') or {}
    if detection.get('strategy') != 'regex':
        raise ValueError(f"Unsupported batch detection strategy: {detection.get('strategy')}")
    pattern = detection.get('pattern')
    if not pattern:
        raise ValueError('Missing batch_detection.pattern')
    match = re.search(pattern, rel_path)
    if not match:
        return None
    return match.group(1).upper()


def iter_source_files(root: Path, include_patterns: list[str], exclude_patterns: list[str]) -> list[Path]:
    if not root.exists():
        raise FileNotFoundError(f'Input root does not exist: {root}')
    if not root.is_dir():
        raise NotADirectoryError(f'Input root must be a directory: {root}')
    files = []
    for path in root.rglob('*'):
        if not path.is_file():
            continue
        rel = path.relative_to(root).as_posix()
        if include_patterns and not matches_any(rel, include_patterns):
            continue
        if exclude_patterns and matches_any(rel, exclude_patterns):
            continue
        files.append(path)
    return sorted(files, key=lambda item: item.as_posix())


def source_dataset_id(source: dict[str, Any]) -> str:
    dataset_id = str(source.get('dataset_id') or source.get('target_dataset_id') or '').strip()
    if not dataset_id:
        raise ValueError(f"Missing dataset_id for source_id={source.get('source_id', '<missing>')}")
    return dataset_id


def required_config_str(source: dict[str, Any], dotted_key: str) -> str:
    value: Any = source
    for key in dotted_key.split('.'):
        if not isinstance(value, dict) or key not in value:
            raise ValueError(f"Missing {dotted_key} for source_id={source.get('source_id', '<missing>')}")
        value = value[key]
    result = str(value).strip()
    if not result:
        raise ValueError(f"Missing {dotted_key} for source_id={source.get('source_id', '<missing>')}")
    return result


def build_gcs_key(raw_prefix: str, dataset_id: str, source_version: str, batch_id: str, rel_path: str) -> str:
    parts = [
        normalize_prefix(raw_prefix),
        f'dataset={dataset_id}',
        f'source_version={source_version}',
        f'batch={batch_id}',
        rel_path.strip('/'),
    ]
    return '/'.join(part for part in parts if part)


def normalize_relative_path(rel_path: str, source: dict[str, Any]) -> str:
    result = rel_path.replace('\\', '/').strip('/')
    for raw_prefix in source.get('relative_path_prefixes_to_strip') or []:
        prefix = str(raw_prefix).replace('\\', '/').strip('/')
        if not prefix:
            continue
        if result.lower() == prefix.lower():
            return ''
        if result.lower().startswith(prefix.lower() + '/'):
            return result[len(prefix):].lstrip('/')
    return result


## 5. Define Artifact, Credential, and Upload Helpers

Create helpers for run directories, logging, Kaggle Secrets, environment fallback, bucket/prefix parsing, GCS client creation, object upload, and JSONL artifact writing.

These functions are separated from the main runner so the upload behavior is easier to inspect and test in small pieces.

In [ ]:
def make_run_paths(base_dir: Path, run_id: str) -> RunPaths:
    run_dir = base_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    return RunPaths(
        run_dir=run_dir,
        manifest=run_dir / 'manifest.jsonl',
        summary=run_dir / 'summary.json',
        errors=run_dir / 'errors.jsonl',
        metrics=run_dir / 'metrics.csv',
        log=run_dir / 'ingest.log',
    )


def setup_logging(log_path: Path, verbose: bool) -> logging.Logger:
    logger = logging.getLogger('kaggle_ingest')
    logger.handlers.clear()
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
    file_handler = logging.FileHandler(log_path, encoding='utf-8')
    file_handler.setFormatter(formatter)
    file_handler.setLevel(logging.DEBUG)
    logger.addHandler(file_handler)
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    console_handler.setLevel(logging.DEBUG if verbose else logging.INFO)
    logger.addHandler(console_handler)
    return logger


def read_kaggle_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or ''
    except Exception:
        return ''


def split_gcs_bucket_value(value: str) -> tuple[str, str]:
    value = str(value or '').strip()
    if not value:
        return '', ''
    if value.startswith('gs://'):
        value = value[len('gs://'):]
    value = value.strip('/')
    bucket, sep, prefix = value.partition('/')
    return bucket.strip(), normalize_prefix(prefix) if sep else ''


def resolve_gcs_destination(args: SimpleNamespace, dry_run: bool, default_raw_prefix: str) -> tuple[str, str]:
    bucket_value = args.gcs_bucket or os.environ.get('GCS_BUCKET', '') or read_kaggle_secret('GCS_BUCKET')
    bucket_name, bucket_prefix = split_gcs_bucket_value(bucket_value)
    explicit_prefix = normalize_prefix(args.gcs_prefix or '')
    if bucket_prefix and explicit_prefix:
        raise ValueError('GCS_BUCKET already includes a path prefix. Leave GCS_PREFIX empty or set GCS_BUCKET to bucket name only.')
    if not bucket_name and not dry_run:
        raise RuntimeError('Set GCS_BUCKET in the config cell, environment, or Kaggle Secret.')
    return bucket_name, explicit_prefix or bucket_prefix or normalize_prefix(default_raw_prefix)


def resolve_credentials(args: SimpleNamespace) -> tuple[str, str]:
    creds_file = args.gcs_credentials_file or os.environ.get('GCS_CREDENTIALS_FILE', '')
    creds_json = os.environ.get('GCS_CREDENTIALS_JSON', '') or read_kaggle_secret('GCS_CREDENTIALS_JSON')
    return creds_file, creds_json


def make_gcs_bucket(bucket_name: str, credentials_file: str, credentials_json: str):
    from google.cloud import storage
    if credentials_json:
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_info(json.loads(credentials_json))
        client = storage.Client(project=credentials.project_id, credentials=credentials)
    elif credentials_file:
        client = storage.Client.from_service_account_json(credentials_file)
    else:
        client = storage.Client()
    return client.bucket(bucket_name)


def upload_one(bucket: Any, record: dict[str, Any], skip_existing: bool, overwrite: bool) -> UploadResult:
    start = time.perf_counter()
    blob = bucket.blob(record['gcs_key'])
    blob.chunk_size = 8 * 1024 * 1024
    try:
        if skip_existing and blob.exists():
            elapsed = int((time.perf_counter() - start) * 1000)
            return UploadResult(status='skipped', bytes_uploaded=0, duration_ms=elapsed, generation=str(blob.generation or ''))
        content_type = mimetypes.guess_type(record['local_path'])[0] or 'application/octet-stream'
        kwargs: dict[str, Any] = {'content_type': content_type, 'timeout': 600}
        if not overwrite:
            kwargs['if_generation_match'] = 0
        blob.upload_from_filename(record['local_path'], **kwargs)
        elapsed = int((time.perf_counter() - start) * 1000)
        return UploadResult(status='uploaded', bytes_uploaded=int(record['size_bytes']), duration_ms=elapsed, generation=str(blob.generation or ''))
    except Exception as exc:
        elapsed = int((time.perf_counter() - start) * 1000)
        return UploadResult(status='failed', bytes_uploaded=0, duration_ms=elapsed, error=str(exc))


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')


## 6. Build Manifest, Metrics, and Summary Outputs

Scan the input root and create the planned upload manifest. Each manifest row includes the local file path, detected batch, normalized relative path, planned GCS key, file size, and upload metadata.

Files that cannot be mapped to an expected batch are written to `unmapped.jsonl`. The same cell also defines metrics CSV output, run artifact uploads, and the final summary payload.

In [ ]:
def discover_manifest(
    source: dict[str, Any],
    input_root: Path,
    selected_batches: set[str],
    bucket_name: str,
    raw_prefix: str,
    source_version: str,
    max_files: int | None,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]], int]:
    include_patterns = list(source.get('include_patterns') or [])
    exclude_patterns = list(source.get('exclude_patterns') or [])
    expected_batches = {batch.upper() for batch in source.get('expected_batches') or []}
    dataset_id = source_dataset_id(source)
    planned = []
    unmapped = []
    filtered_by_batch = 0
    for path in iter_source_files(input_root, include_patterns, exclude_patterns):
        rel = path.relative_to(input_root).as_posix()
        batch_id = detect_batch(rel, source)
        size_bytes = path.stat().st_size
        if not batch_id or batch_id not in expected_batches:
            unmapped.append({'relative_path': rel, 'local_path': str(path), 'size_bytes': size_bytes, 'reason': 'batch_not_detected_or_unexpected'})
            continue
        if batch_id not in selected_batches:
            filtered_by_batch += 1
            continue
        normalized_rel = normalize_relative_path(rel, source)
        gcs_key = build_gcs_key(raw_prefix, dataset_id, source_version, batch_id, normalized_rel)
        planned.append({
            'manifest_schema_version': 1,
            'source_id': source['source_id'],
            'source_type': source.get('source_type', 'kaggle'),
            'dataset_ref': source.get('dataset_ref', ''),
            'dataset_id': dataset_id,
            'batch_id': batch_id,
            'source_version': source_version,
            'relative_path': normalized_rel,
            'source_relative_path': rel,
            'local_path': str(path),
            'filename': path.name,
            'extension': path.suffix.lower(),
            'size_bytes': size_bytes,
            'gcs_bucket': bucket_name,
            'gcs_key': gcs_key,
            'gcs_uri': f'gs://{bucket_name}/{gcs_key}' if bucket_name else '',
            'planned_at': utc_now_iso(),
        })
    if max_files is not None:
        planned = planned[:max_files]
    return planned, unmapped, filtered_by_batch


def write_initial_metrics(metrics_path: Path) -> csv.DictWriter:
    handle = metrics_path.open('w', encoding='utf-8', newline='')
    fieldnames = ['ts', 'run_id', 'source_id', 'batch_id', 'relative_path', 'gcs_uri', 'status', 'size_bytes', 'bytes_uploaded', 'duration_ms', 'generation', 'error']
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer._handle = handle
    return writer


def close_metrics_writer(writer: csv.DictWriter) -> None:
    handle = getattr(writer, '_handle', None)
    if handle:
        handle.close()


def artifact_gcs_keys(source: dict[str, Any], run_id: str) -> dict[str, str]:
    control_prefix = normalize_prefix(required_config_str(source, 'gcs.control_prefix'))
    logs_prefix = normalize_prefix(required_config_str(source, 'gcs.logs_prefix'))
    return {
        'manifest': f'{control_prefix}/run_id={run_id}/manifest.jsonl',
        'errors': f'{control_prefix}/run_id={run_id}/errors.jsonl',
        'metrics': f'{logs_prefix}/run_id={run_id}/metrics.csv',
        'log': f'{logs_prefix}/run_id={run_id}/ingest.log',
        'summary': f'{control_prefix}/run_id={run_id}/summary.json',
    }


def artifact_upload_mappings(source: dict[str, Any], paths: RunPaths) -> list[tuple[Path, str]]:
    keys = artifact_gcs_keys(source, paths.run_dir.name)
    return [
        (paths.manifest, keys['manifest']),
        (paths.errors, keys['errors']),
        (paths.metrics, keys['metrics']),
        (paths.log, keys['log']),
        (paths.summary, keys['summary']),
    ]


def artifact_paths(paths: RunPaths, include_unmapped: bool) -> dict[str, str]:
    result = {
        'manifest': str(paths.manifest),
        'summary': str(paths.summary),
        'errors': str(paths.errors),
        'metrics': str(paths.metrics),
        'log': str(paths.log),
    }
    if include_unmapped:
        result['unmapped'] = str(paths.run_dir / 'unmapped.jsonl')
    return result


def artifact_uris(source: dict[str, Any], run_id: str, bucket_name: str) -> dict[str, str]:
    if not bucket_name:
        return {}
    return {name: f'gs://{bucket_name}/{key}' for name, key in artifact_gcs_keys(source, run_id).items()}


def upload_run_artifacts(bucket: Any, source: dict[str, Any], paths: RunPaths, logger: logging.Logger) -> None:
    for local_path, key in artifact_upload_mappings(source, paths):
        blob = bucket.blob(key)
        content_type = mimetypes.guess_type(local_path.name)[0] or 'application/octet-stream'
        blob.upload_from_filename(str(local_path), content_type=content_type)
        logger.info('uploaded run artifact gs://%s/%s', bucket.name, key)


def run_status(args: SimpleNamespace, failed: int, unmapped_count: int, artifact_upload_status: str) -> str:
    if failed:
        return 'failed'
    if artifact_upload_status.startswith('failed'):
        return 'completed_artifact_upload_failed'
    if args.dry_run:
        return 'dry_run'
    if unmapped_count:
        return 'completed_with_unmapped'
    return 'completed'


def summarize(
    run_id: str,
    args: SimpleNamespace,
    source: dict[str, Any],
    planned: list[dict[str, Any]],
    unmapped: list[dict[str, Any]],
    filtered_by_batch: int,
    uploaded: int,
    skipped: int,
    failed: int,
    bytes_uploaded: int,
    started_at: float,
    artifact_upload_status: str,
) -> dict[str, Any]:
    per_batch: dict[str, dict[str, int]] = {}
    for row in planned:
        entry = per_batch.setdefault(row['batch_id'], {
            'planned': 0,
            'planned_files': 0,
            'size_bytes': 0,
            'bytes_planned': 0,
            'uploaded': 0,
            'uploaded_files': 0,
            'skipped': 0,
            'skipped_files': 0,
            'failed': 0,
            'failed_files': 0,
            'bytes_uploaded': 0,
            'duration_ms': 0,
        })
        entry['planned'] += 1
        entry['planned_files'] += 1
        entry['size_bytes'] += int(row['size_bytes'])
        entry['bytes_planned'] += int(row['size_bytes'])
        status = row.get('upload_status')
        if status == 'uploaded':
            entry['uploaded'] += 1
            entry['uploaded_files'] += 1
        elif status == 'skipped':
            entry['skipped'] += 1
            entry['skipped_files'] += 1
        elif status == 'failed':
            entry['failed'] += 1
            entry['failed_files'] += 1
        entry['bytes_uploaded'] += int(row.get('bytes_uploaded', 0) or 0)
        entry['duration_ms'] += int(row.get('duration_ms', 0) or 0)

    elapsed_s = round(time.perf_counter() - started_at, 3)
    planned_count = len(planned)
    success_count = uploaded + skipped
    success_rate = round(success_count / planned_count, 6) if planned_count else 0.0
    failure_rate = round(failed / planned_count, 6) if planned_count else 0.0
    bucket_name = getattr(args, 'resolved_bucket_name', '')
    paths = make_run_paths(args.run_dir, run_id)
    return {
        'monitoring_schema_version': 2,
        'run_id': run_id,
        'run_status': run_status(args, failed, len(unmapped), artifact_upload_status),
        'started_at': args.started_at,
        'finished_at': utc_now_iso(),
        'elapsed_seconds': elapsed_s,
        'dry_run': args.dry_run,
        'source_id': source['source_id'],
        'source_type': source.get('source_type', 'kaggle'),
        'dataset_ref': source.get('dataset_ref', ''),
        'dataset_id': source_dataset_id(source),
        'input_root': str(args.resolved_input_root),
        'gcs_bucket': bucket_name,
        'raw_prefix': getattr(args, 'resolved_raw_prefix', ''),
        'selected_batches': sorted(args.selected_batches),
        'workers': args.workers,
        'skip_existing': args.skip_existing,
        'overwrite': args.overwrite,
        'planned_files': planned_count,
        'unmapped_files': len(unmapped),
        'filtered_by_batch_files': filtered_by_batch,
        'uploaded_files': uploaded,
        'skipped_files': skipped,
        'failed_files': failed,
        'bytes_planned': sum(int(row['size_bytes']) for row in planned),
        'bytes_uploaded': bytes_uploaded,
        'success_rate': success_rate,
        'failure_rate': failure_rate,
        'per_batch': per_batch,
        'artifact_upload_status': artifact_upload_status,
        'artifact_paths': artifact_paths(paths, bool(unmapped)),
        'artifact_uris': artifact_uris(source, run_id, bucket_name),
    }

## 7. Orchestrate One Ingestion Run

Run the full ingestion workflow: validate settings, select the source and batches, resolve the input root and GCS destination, create manifest artifacts, perform either a dry run or a real upload, then write metrics, errors, and summary files.

When `DRY_RUN = True`, no files are uploaded. The notebook only writes local artifacts so paths and counts can be checked safely.

In [ ]:
def run(args: SimpleNamespace) -> int:
    args.started_at = utc_now_iso()
    started_at = time.perf_counter()
    config = args.config_data
    if args.list_sources:
        list_sources(config)
        return 0
    if not args.source_id:
        raise ValueError('SOURCE_ID is required unless list_sources is used.')
    if args.workers < 1:
        raise ValueError('WORKERS must be >= 1')
    if args.max_files is not None and args.max_files < 1:
        raise ValueError('MAX_FILES must be >= 1')
    if args.overwrite and args.skip_existing:
        raise ValueError('OVERWRITE conflicts with SKIP_EXISTING. Set SKIP_EXISTING = False when OVERWRITE = True.')

    source = select_source(config, args.source_id)
    args.selected_batches = parse_batches(args.batches, list(source.get('expected_batches') or []))
    input_root = args.input_root or Path(required_config_str(source, 'kaggle_mount_path'))
    args.resolved_input_root = input_root
    source_version = args.source_version or required_config_str(source, 'source_version')
    default_raw_prefix = required_config_str(source, 'gcs.raw_prefix')
    bucket_name, raw_prefix = resolve_gcs_destination(args, args.dry_run, default_raw_prefix)
    args.resolved_bucket_name = bucket_name
    args.resolved_raw_prefix = normalize_prefix(raw_prefix)

    run_id = args.run_id or new_run_id()
    paths = make_run_paths(args.run_dir, run_id)
    logger = setup_logging(paths.log, args.verbose)
    logger.info('run_id=%s source_id=%s input_root=%s dry_run=%s', run_id, source['source_id'], input_root, args.dry_run)
    logger.info('target bucket=%s raw_prefix=%s', bucket_name or '<not set>', raw_prefix)
    logger.info('selected batches: %s', ','.join(sorted(args.selected_batches)))

    planned, unmapped, filtered_by_batch = discover_manifest(
        source=source,
        input_root=input_root,
        selected_batches=args.selected_batches,
        bucket_name=bucket_name,
        raw_prefix=raw_prefix,
        source_version=source_version,
        max_files=args.max_files,
    )
    for row in planned:
        row['run_id'] = run_id
    write_jsonl(paths.manifest, planned)
    write_jsonl(paths.errors, [])
    logger.info('planned=%d unmapped=%d filtered_by_batch=%d', len(planned), len(unmapped), filtered_by_batch)
    if unmapped:
        unmapped_path = paths.run_dir / 'unmapped.jsonl'
        write_jsonl(unmapped_path, unmapped)
        logger.warning('wrote unmapped report: %s', unmapped_path)
        if args.fail_on_unmapped:
            raise RuntimeError(f'{len(unmapped)} included file(s) could not be mapped to an expected batch.')
    if not planned:
        raise RuntimeError('No files planned. Check INPUT_ROOT, include patterns, and BATCHES.')

    uploaded = skipped = failed = bytes_uploaded = 0
    artifact_upload_status = 'not_uploaded'
    writer = write_initial_metrics(paths.metrics)
    errors_handle = paths.errors.open('a', encoding='utf-8')
    try:
        if args.dry_run:
            for row in planned:
                row['upload_status'] = 'planned'
                row['bytes_uploaded'] = 0
                row['duration_ms'] = 0
                writer.writerow({
                    'ts': utc_now_iso(),
                    'run_id': run_id,
                    'source_id': source['source_id'],
                    'batch_id': row['batch_id'],
                    'relative_path': row['relative_path'],
                    'gcs_uri': row['gcs_uri'],
                    'status': 'planned',
                    'size_bytes': row['size_bytes'],
                    'bytes_uploaded': 0,
                    'duration_ms': 0,
                    'generation': '',
                    'error': '',
                })
            artifact_upload_status = 'dry_run'
        else:
            credentials_file, credentials_json = resolve_credentials(args)
            bucket = make_gcs_bucket(bucket_name, credentials_file, credentials_json)
            progress = None
            if not args.no_progress:
                try:
                    from tqdm import tqdm
                    progress = tqdm(total=len(planned), unit='file')
                except ImportError:
                    progress = None
            with ThreadPoolExecutor(max_workers=args.workers) as pool:
                futures = {pool.submit(upload_one, bucket, row, args.skip_existing, args.overwrite): row for row in planned}
                for future in as_completed(futures):
                    row = futures[future]
                    result = future.result()
                    if result.status == 'uploaded':
                        uploaded += 1
                        bytes_uploaded += result.bytes_uploaded
                    elif result.status == 'skipped':
                        skipped += 1
                    else:
                        failed += 1
                        error_row = {**row, 'status': result.status, 'error': result.error, 'failed_at': utc_now_iso()}
                        errors_handle.write(json.dumps(error_row, ensure_ascii=False, sort_keys=True) + '\n')

                    row['upload_status'] = result.status
                    row['bytes_uploaded'] = result.bytes_uploaded
                    row['duration_ms'] = result.duration_ms
                    row['generation'] = result.generation
                    row['error'] = result.error
                    writer.writerow({
                        'ts': utc_now_iso(),
                        'run_id': run_id,
                        'source_id': source['source_id'],
                        'batch_id': row['batch_id'],
                        'relative_path': row['relative_path'],
                        'gcs_uri': row['gcs_uri'],
                        'status': result.status,
                        'size_bytes': row['size_bytes'],
                        'bytes_uploaded': result.bytes_uploaded,
                        'duration_ms': result.duration_ms,
                        'generation': result.generation,
                        'error': result.error,
                    })
                    if progress:
                        progress.set_postfix(uploaded=uploaded, skipped=skipped, failed=failed, refresh=False)
                        progress.update(1)
            if progress:
                progress.close()
            if args.upload_run_artifacts:
                close_metrics_writer(writer)
                errors_handle.close()
                summary = summarize(run_id, args, source, planned, unmapped, filtered_by_batch, uploaded, skipped, failed, bytes_uploaded, started_at, 'uploaded')
                paths.summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True), encoding='utf-8')
                try:
                    upload_run_artifacts(bucket, source, paths, logger)
                    artifact_upload_status = 'uploaded'
                except Exception as exc:
                    artifact_upload_status = f'failed: {exc}'
                    logger.error('run artifact upload failed: %s', exc)
                writer = None
                errors_handle = None
    finally:
        if writer is not None:
            close_metrics_writer(writer)
        if errors_handle is not None:
            errors_handle.close()

    summary = summarize(run_id, args, source, planned, unmapped, filtered_by_batch, uploaded, skipped, failed, bytes_uploaded, started_at, artifact_upload_status)
    paths.summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True), encoding='utf-8')
    logger.info('finished planned=%d uploaded=%d skipped=%d failed=%d bytes_uploaded=%d run_dir=%s', len(planned), uploaded, skipped, failed, bytes_uploaded, paths.run_dir)
    print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))
    return 1 if failed else 0

## 8. Build Runtime Arguments and Preview Sources

Collect the configuration variables into a `SimpleNamespace` object for `run()`. The final `list_sources(CONFIG)` call prints the configured sources, batches, and Kaggle mount paths before execution.

In [ ]:
def build_args_from_config() -> SimpleNamespace:
    return SimpleNamespace(
        config_data=CONFIG,
        list_sources=False,
        source_id=SOURCE_ID,
        batches=BATCHES,
        input_root=Path(INPUT_ROOT) if INPUT_ROOT else None,
        gcs_bucket=GCS_BUCKET,
        gcs_prefix=GCS_PREFIX,
        gcs_credentials_file=GCS_CREDENTIALS_FILE,
        source_version=SOURCE_VERSION,
        workers=WORKERS,
        run_id=RUN_ID,
        run_dir=Path(RUN_DIR),
        max_files=MAX_FILES,
        dry_run=DRY_RUN,
        skip_existing=SKIP_EXISTING,
        overwrite=OVERWRITE,
        upload_run_artifacts=UPLOAD_RUN_ARTIFACTS,
        fail_on_unmapped=FAIL_ON_UNMAPPED,
        no_progress=NO_PROGRESS,
        verbose=VERBOSE,
    )


list_sources(CONFIG)


## 9. Execute the Pipeline

Run the pipeline with the current configuration. First use `DRY_RUN = True` and a small `MAX_FILES` value to inspect generated `gcs_key` values. After the manifest looks correct, set `DRY_RUN = False` to upload files.

In [ ]:
# Recommended first run: keep DRY_RUN = True and set MAX_FILES = 1 in the config cell.
args = build_args_from_config()
exit_code = run(args)
print(f'Exit code: {exit_code}')


## 10. Monitoring Handoff

Read the latest `summary.json` generated by the run and display the operational fields that the Prometheus exporter will also expose. This does not start Prometheus inside Kaggle; it only verifies that the artifacts are ready for the cloud monitoring service.

In [ ]:
summary_files = sorted(Path(RUN_DIR).glob('*/summary.json'), key=lambda path: path.stat().st_mtime, reverse=True)
if not summary_files:
    raise FileNotFoundError(f'No summary.json found under {RUN_DIR}')

latest_summary_path = summary_files[0]
latest_summary = json.loads(latest_summary_path.read_text(encoding='utf-8'))

run_overview = {
    'run_id': latest_summary.get('run_id'),
    'run_status': latest_summary.get('run_status'),
    'dry_run': latest_summary.get('dry_run'),
    'source_id': latest_summary.get('source_id'),
    'dataset_id': latest_summary.get('dataset_id'),
    'planned_files': latest_summary.get('planned_files'),
    'uploaded_files': latest_summary.get('uploaded_files'),
    'skipped_files': latest_summary.get('skipped_files'),
    'failed_files': latest_summary.get('failed_files'),
    'unmapped_files': latest_summary.get('unmapped_files'),
    'bytes_uploaded': latest_summary.get('bytes_uploaded'),
    'success_rate': latest_summary.get('success_rate'),
    'artifact_upload_status': latest_summary.get('artifact_upload_status'),
}

print('Latest summary:', latest_summary_path)
print(json.dumps(run_overview, indent=2, ensure_ascii=False))

per_batch_rows = []
for batch_id, stats in (latest_summary.get('per_batch') or {}).items():
    per_batch_rows.append({'batch_id': batch_id, **stats})

try:
    import pandas as pd
    display(pd.DataFrame([run_overview]))
    if per_batch_rows:
        display(pd.DataFrame(per_batch_rows).sort_values('batch_id'))
except Exception:
    print('Per-batch summary:')
    print(json.dumps(per_batch_rows, indent=2, ensure_ascii=False))

print('\nLocal artifact paths:')
for name, value in (latest_summary.get('artifact_paths') or {}).items():
    print(f'- {name}: {value}')

print('\nGCS artifact URIs for the Prometheus exporter:')
for name, value in (latest_summary.get('artifact_uris') or {}).items():
    print(f'- {name}: {value}')

print('\nExporter metrics to expect:')
for metric_name in [
    'kaggle_ingest_run_info',
    'kaggle_ingest_run_elapsed_seconds',
    'kaggle_ingest_files_total',
    'kaggle_ingest_bytes_total',
    'kaggle_ingest_batch_files_total',
    'kaggle_ingest_batch_bytes_total',
    'kaggle_ingest_unmapped_files',
    'kaggle_ingest_artifact_upload_status',
    'kaggle_ingest_last_success_timestamp_seconds',
]:
    print(f'- {metric_name}')